# Course 2 · Enterprise Track 2b · The Next Delivery

**Goal:** process a second delivery incrementally, with the pipeline course 1 built. Course 1's `02_pipeline` ended by handing over the next deliveries in a handoff folder of your volume; course 2 lands them, checking each fingerprint first, and never runs course 1's generator.

> ⚠️ **Mock data only.** This track runs on the agency workspace, which is approved for criminal justice information, but every table here holds the course's fictional records. Real tables come later, with the data owner's approval (see `enterprise/README.md`).

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds course 2's helpers and checks your settings.
STEP = 2

import importlib
import sys
from pathlib import Path

def _find_track():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # course 2's enterprise/ folder in its Git folder
        if (folder / "helpers.py").exists() and (folder / "settings.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "helpers.py").exists() and (folder / "settings.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find course 2's helpers. Open this notebook from inside course 2's Git folder (enterprise/).")

TRACK = _find_track()
sys.path.insert(0, str(TRACK))
import settings
import helpers as ent
importlib.reload(settings)                                       # picks up edits to settings.py without a restart
importlib.reload(ent)
N = ent.prepare(STEP, "The next delivery")

## 1. The next delivery: incremental processing
A day later the source drops another file. Land it from the handoff (the fingerprint must match what course 1 recorded), then run course 1's pipeline. The streaming tables process **only the new file**: the 100,000 rows from batch 1 aren't read again. Watch `_source_file`.

In [ ]:
pipeline_id = ent.find_pipeline(N)        # course 1's pipeline, by name
ent.land_delivery(N, "batch_002.csv")
second = ent.run_pipeline(pipeline_id)
display(ent.sql(f"SELECT _source_file, count(*) AS records FROM {N['fq']}.silver_offenders GROUP BY ALL ORDER BY 1"))

Two things to know:
- **A crosswalk change doesn't re-map rows already in silver.** Streaming tables process each row once. After a data steward changes reference data, run a **full refresh** (`ent.run_pipeline(pipeline_id, full_refresh=True)`) so every row is mapped with the new version.
- **Development vs production mode.** Development mode is for iterating. The job in 04 runs the same pipeline the way production would: on a schedule, with retries and alerts.
- **An expectation on a materialized view means a full recompute.** `gold_people_by_unit` has one, so it's rebuilt on every update instead of incrementally. That's fine at this size; on a large table, put the check in `validate_quality` instead.

**Next:** 03_governance controls who sees race and religion.